In [ ]:
import numpy as np
import pandas as pd
import re
import string
import joblib

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Optional boosting libraries - the notebook still runs fine without them,
# those two pipelines are just skipped.
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed - skipping those pipelines. Install with: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not installed - skipping those pipelines. Install with: pip install lightgbm")


In [ ]:
df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
print(df.shape)
df.head()

In [ ]:
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).round(3))

In [ ]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)      # strip URLs
    text = re.sub(r'\d+', ' ', text)                  # strip numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(clean_text)
df[['message', 'clean_message']].head()

In [ ]:
X = df['clean_message']
y = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
def get_vectorizers():
    return {
        "TF-IDF": TfidfVectorizer(max_features=3000),
        "CountVectorizer": CountVectorizer(max_features=3000),
    }

In [ ]:
def get_models():
    models = {
        "Perceptron": Perceptron(max_iter=1000, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Linear SVM": LinearSVC(max_iter=5000),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
        "Gradient Boosting (sklearn)": GradientBoostingClassifier(random_state=42),
        "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(eval_metric='logloss', random_state=42)
    if HAS_LGBM:
        models["LightGBM"] = LGBMClassifier(random_state=42, verbose=-1)
    return models

In [ ]:
all_results = []
trained_pipelines = {}

for vec_name, vectorizer in get_vectorizers().items():
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    for model_name, model in get_models().items():
        key = f"{model_name} | {vec_name}"
        try:
            try:
                model.fit(X_train_vec, y_train)
                preds = model.predict(X_test_vec)
            except TypeError:
                # A few estimators don't accept sparse input - fall back to dense
                model.fit(X_train_vec.toarray(), y_train)
                preds = model.predict(X_test_vec.toarray())

            acc = accuracy_score(y_test, preds)
            prec = precision_score(y_test, preds, zero_division=0)
            rec = recall_score(y_test, preds, zero_division=0)
            f1 = f1_score(y_test, preds, zero_division=0)

            trained_pipelines[key] = {
                "model": model,
                "vectorizer": vectorizer,
                "vectorizer_name": vec_name,
                "algorithm": model_name,
            }

            all_results.append({
                "Pipeline Key": key,
                "Algorithm": model_name,
                "Vectorizer": vec_name,
                "Accuracy": round(acc, 4),
                "Precision": round(prec, 4),
                "Recall": round(rec, 4),
                "F1 Score": round(f1, 4),
            })

            print(f"OK   {key}  (Acc={acc:.4f}, F1={f1:.4f})")

        except Exception as e:
            print(f"FAIL {key}  -> {e}")

results_df = (
    pd.DataFrame(all_results)
    .sort_values(by="F1 Score", ascending=False)
    .reset_index(drop=True)
)

best_pipeline_key = results_df.iloc[0]["Pipeline Key"]

print("\n" + "=" * 60)
print("BEST PIPELINE")
print("=" * 60)
print(best_pipeline_key)
print(results_df.iloc[0])

In [ ]:
results_df

In [ ]:
# Lock in the single best pipeline - this is the ONLY one the app will ever use.
best_entry = trained_pipelines[best_pipeline_key]
vectorizer = best_entry["vectorizer"]
model = best_entry["model"]

# Some strong classifiers (Linear SVM, Perceptron) don't output probabilities
# by default. If the winner is one of those, wrap it in a calibrator so the
# UI can still show a genuine confidence score instead of just a hard label.
if hasattr(model, "predict_proba"):
    production_model = model
else:
    print(f"{best_entry['algorithm']} has no predict_proba - calibrating for confidence scores...")
    X_train_vec = vectorizer.transform(X_train)
    production_model = CalibratedClassifierCV(clone(model), cv=3, method="sigmoid")
    production_model.fit(X_train_vec, y_train)

production_pipeline = {
    "model": production_model,
    "vectorizer": vectorizer,
    "algorithm": best_entry["algorithm"],
    "vectorizer_name": best_entry["vectorizer_name"],
}

print(f"Production pipeline locked in: {best_pipeline_key}")

In [ ]:
bundle = {
    "production_pipeline": production_pipeline,   # the one pipeline the app runs
    "results_df": results_df,                      # kept only for the read-only Model Info tab
    "best_pipeline_key": best_pipeline_key,
}

joblib.dump(bundle, "spam_pipeline_bundle.pkl")
print("Saved spam_pipeline_bundle.pkl")
print(f"Locked-in pipeline: {best_pipeline_key}")
print(f"F1 Score: {results_df.iloc[0]['F1 Score']}  |  Accuracy: {results_df.iloc[0]['Accuracy']}")

In [ ]:
import html as html_lib

SPAM_KEYWORDS = {
    "free", "win", "winner", "won", "cash", "prize", "urgent", "click",
    "claim", "credit", "guarantee", "guaranteed", "offer", "txt", "limited",
    "congratulations", "voucher", "subscribe", "trial", "bonus", "reward",
    "selected", "exclusive", "cheap", "discount", "deal", "instant",
    "million", "loan", "collect", "award", "chance", "entry", "membership",
    "ringtone", "call", "mobile", "camera",
}


def highlight_keywords(message):
    """Wraps likely spam trigger words in a highlighted <mark> span."""
    escaped = html_lib.escape(message)
    tokens = escaped.split(" ")
    highlighted = []
    for tok in tokens:
        bare = re.sub(r"[^a-zA-Z]", "", tok).lower()
        if bare in SPAM_KEYWORDS:
            highlighted.append(
                f'<mark style="background:#ffd7d7;color:#8a0000;font-weight:600;'
                f'border-radius:3px;padding:0 2px;">{tok}</mark>'
            )
        else:
            highlighted.append(tok)
    return '<div style="font-size:15px;line-height:1.7;">' + " ".join(highlighted) + "</div>"


def message_stats(message):
    """Quick heuristic signals that are known to correlate with spam."""
    words = message.split()
    stats = {
        "Characters": len(message),
        "Words": len(words),
        "ALL-CAPS words": sum(1 for w in words if len(w) > 1 and w.isupper()),
        "Exclamation marks": message.count("!"),
        "Digits": sum(ch.isdigit() for ch in message),
        "Currency symbols": sum(message.count(s) for s in ["£", "$", "€"]),
        "Contains URL": "Yes" if re.search(r"http\S+|www\S+", message.lower()) else "No",
    }
    return pd.DataFrame(list(stats.items()), columns=["Signal", "Value"])

In [ ]:
import joblib
import pandas as pd
import gradio as gr
import plotly.express as px

bundle = joblib.load("spam_pipeline_bundle.pkl")
production_pipeline = bundle["production_pipeline"]
results_df = bundle["results_df"]

vectorizer = production_pipeline["vectorizer"]
production_model = production_pipeline["model"]

model_info_line = (
    f"**Model in use:** {production_pipeline['algorithm']} "
    f"({production_pipeline['vectorizer_name']}) &nbsp;|&nbsp; "
    f"**F1:** {results_df.iloc[0]['F1 Score']} &nbsp;|&nbsp; "
    f"**Accuracy:** {results_df.iloc[0]['Accuracy']}"
)


def classify_message(message):
    if not message or not message.strip():
        empty_stats = pd.DataFrame(columns=["Signal", "Value"])
        return {"HAM": 0.5, "SPAM": 0.5}, "<i>Type a message above to see it analyzed here.</i>", empty_stats, "—"

    cleaned = clean_text(message)
    X_input = vectorizer.transform([cleaned])
    proba = production_model.predict_proba(X_input)[0]
    spam_prob = float(proba[1])
    label = "SPAM" if spam_prob >= 0.5 else "HAM"

    confidences = {"SPAM": spam_prob, "HAM": 1 - spam_prob}
    highlighted = highlight_keywords(message)
    stats_df = message_stats(message)

    return confidences, highlighted, stats_df, label


def classify_and_log(message, history):
    confidences, highlighted, stats_df, label = classify_message(message)
    history = history or []
    if message and message.strip():
        preview = message.strip()
        preview = preview[:60] + "..." if len(preview) > 60 else preview
        history.insert(0, {
            "Message": preview,
            "Prediction": label,
            "Spam Probability": f"{confidences['SPAM']*100:.1f}%",
        })
        history = history[:10]
    history_df = pd.DataFrame(history) if history else pd.DataFrame(
        columns=["Message", "Prediction", "Spam Probability"]
    )
    return confidences, highlighted, stats_df, history, history_df


def batch_classify(file, pasted_text):
    messages = []
    if file is not None:
        try:
            if str(file.name).lower().endswith(".csv"):
                bdf = pd.read_csv(file.name)
                col = "message" if "message" in bdf.columns else bdf.columns[0]
                messages = bdf[col].astype(str).tolist()
            else:
                with open(file.name, "r", encoding="utf-8", errors="ignore") as f:
                    messages = [line.strip() for line in f if line.strip()]
        except Exception as e:
            return pd.DataFrame({"Error": [str(e)]}), None
    elif pasted_text and pasted_text.strip():
        messages = [line.strip() for line in pasted_text.split("\n") if line.strip()]

    if not messages:
        empty = pd.DataFrame(columns=["Message", "Prediction", "Spam Probability"])
        return empty, None

    cleaned = [clean_text(m) for m in messages]
    X_batch = vectorizer.transform(cleaned)
    probs = production_model.predict_proba(X_batch)[:, 1]

    results = pd.DataFrame({
        "Message": messages,
        "Prediction": ["SPAM" if p >= 0.5 else "HAM" for p in probs],
        "Spam Probability": [f"{p*100:.1f}%" for p in probs],
    })

    out_path = "batch_results.csv"
    results.to_csv(out_path, index=False)
    return results, out_path


with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"), title="SMS Spam Classifier") as app:
    gr.Markdown("# SMS Spam Classifier")
    gr.Markdown(model_info_line)
    gr.Markdown("This app always runs the single best-performing pipeline found during training — there's no model to pick.")

    with gr.Tab("Classify"):
        history_state = gr.State([])

        message_input = gr.Textbox(lines=4, label="SMS Message", placeholder="Type or paste a message...")
        classify_btn = gr.Button("Classify Message", variant="primary")

        with gr.Row():
            confidence_output = gr.Label(label="Prediction Confidence", num_top_classes=2)
            stats_output = gr.Dataframe(label="Message Signals", interactive=False)

        highlighted_output = gr.HTML(label="Flagged Words")

        gr.Markdown("### Recent classifications (this session)")
        history_table = gr.Dataframe(interactive=False)

        classify_btn.click(
            fn=classify_and_log,
            inputs=[message_input, history_state],
            outputs=[confidence_output, highlighted_output, stats_output, history_state, history_table],
        )
        message_input.submit(
            fn=classify_and_log,
            inputs=[message_input, history_state],
            outputs=[confidence_output, highlighted_output, stats_output, history_state, history_table],
        )

    with gr.Tab("Batch Classify"):
        gr.Markdown(
            "Upload a `.csv` (with a `message` column) or `.txt` file (one message per line), "
            "or paste messages below (one per line)."
        )
        with gr.Row():
            batch_file = gr.File(label="Upload file", file_types=[".csv", ".txt"])
            batch_text = gr.Textbox(lines=8, label="...or paste messages here (one per line)")
        batch_btn = gr.Button("Classify All", variant="primary")
        batch_output = gr.Dataframe(label="Results", interactive=False)
        batch_download = gr.File(label="Download results as CSV")

        batch_btn.click(
            fn=batch_classify,
            inputs=[batch_file, batch_text],
            outputs=[batch_output, batch_download],
        )

    with gr.Tab("Model Info"):
        gr.Markdown(
            "Shown **for transparency only** — this explains why this pipeline was selected "
            "during training. The app itself always uses the top-ranked pipeline automatically."
        )
        with gr.Row():
            gr.Textbox(value=results_df.iloc[0]["Algorithm"], label="Chosen Algorithm", interactive=False)
            gr.Textbox(value=results_df.iloc[0]["Vectorizer"], label="Chosen Vectorizer", interactive=False)
            gr.Textbox(value=f"{results_df.iloc[0]['F1 Score']}", label="F1 Score", interactive=False)

        fig = px.bar(
            results_df.sort_values("F1 Score", ascending=True),
            x="F1 Score",
            y="Pipeline Key",
            color="Vectorizer",
            orientation="h",
            title="All Trained Pipelines Ranked by F1 Score",
        )
        gr.Plot(fig)

        with gr.Accordion("Full performance table", open=False):
            gr.Dataframe(results_df, interactive=False)

if __name__ == "__main__":
    app.launch(inline=True)